# Environment setup

Create the notebook environment from the `lakehouse_data` directory:

```bash
conda env create -f environment.yml
conda activate local-lakehouse
```

Select the `local-lakehouse` kernel in VS Code. From the repository root, start
Spark Connect, Hive Metastore, and MinIO:

```bash
docker compose up -d --build
```

The Spark Connect server provides Delta, Hadoop S3A, and MinIO credentials. The
notebook only needs the matching Spark Connect Python client.

# Start a spark Session with a catalog

In [5]:
from datetime import datetime, timezone

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .remote("sc://localhost:15002")
    .appName("notebook-delta-streaming")
    .getOrCreate()
)

database = "notebook_connect"
run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
source_table = f"{database}.source_{run_id}"
target_table = f"{database}.target_{run_id}"
checkpoint_location = (
    f"s3a://warehouse/notebook_delta_streaming_checkpoints/{run_id}"
)

spark.sql(
    f"CREATE DATABASE IF NOT EXISTS {database} "
    "LOCATION 's3a://warehouse/notebook_delta_streaming'"
)

initial_data = spark.createDataFrame(
    [(1, "Ada", "engineering"), (2, "Grace", "data")],
    ["id", "name", "team"],
)
(
    initial_data.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(source_table)
)

spark.createDataFrame([], initial_data.schema).write.format("delta").saveAsTable(
    target_table
)

print(f"Source: {source_table}")
print(f"Target: {target_table}")
print(f"Checkpoint: {checkpoint_location}")
print("Source table, initial CDF-enabled rows:")
spark.table(source_table).show()

Source: notebook_connect.source_20261003124824
Target: notebook_connect.target_20261003124824
Checkpoint: s3a://warehouse/notebook_delta_streaming_checkpoints/20261003124824
Source table, initial CDF-enabled rows:
+---+-----+-----------+
| id| name|       team|
+---+-----+-----------+
|  1|  Ada|engineering|
|  2|Grace|       data|
+---+-----+-----------+



# Process the initial CDF

In [6]:
from delta_streaming import (
    CDFStreamingPipeline,
    ChangeTypeFilter,
    DeltaCDFMergeSink,
    DeltaCDFSource,
    IdentityTransformer,
    StreamConfig,
)

config = StreamConfig(
    source_table=source_table,
    target_table=target_table,
    checkpoint_location=checkpoint_location,
    query_name=f"customer-cdf-{run_id}",
    starting_version=0,
)

pipeline = CDFStreamingPipeline(
    source=DeltaCDFSource(config),
    transformer=IdentityTransformer(),
    change_filter=ChangeTypeFilter(["insert", "update_postimage"]),
    sink=DeltaCDFMergeSink(
        spark=spark,
        merge_condition="target.id = source.id",
    ),
)

def run_streaming_step(step_name):
    query = pipeline.start(spark=spark, config=config)
    query.awaitTermination()

    progress = query.lastProgress
    print(f"{step_name} - CDF offsets:")
    if progress:
        for source_progress in progress.get("sources", []):
            start_offset = source_progress.get("startOffset")
            end_offset = source_progress.get("endOffset")
            print(f"  startOffset: {start_offset}")
            print(f"  endOffset: {end_offset}")
            print(f"  checkpoint resume watermark: {end_offset}")
    else:
        print("  No source progress was reported for this run.")

    print(f"{step_name} - target table:")
    spark.table(target_table).orderBy("id").show()


run_streaming_step("Initial batch")

Initial batch - CDF offsets:
  startOffset: None
  endOffset: {"sourceVersion":1,"reservoirId":"9aa080b3-61fd-4c7f-babb-47774c7fdedd","reservoirVersion":1,"index":-1,"isStartingVersion":false}
  checkpoint resume watermark: {"sourceVersion":1,"reservoirId":"9aa080b3-61fd-4c7f-babb-47774c7fdedd","reservoirVersion":1,"index":-1,"isStartingVersion":false}
Initial batch - target table:
+---+-----+-----------+
| id| name|       team|
+---+-----+-----------+
|  1|  Ada|engineering|
|  2|Grace|       data|
+---+-----+-----------+



# Add more data and resume from the checkpoint

This changes Ada's row and inserts Linus. The next `availableNow` run uses the
same checkpoint, so it processes only the new CDF version.

In [3]:
spark.sql(
    f"UPDATE {source_table} "
    "SET name = 'Ada Lovelace' WHERE id = 1"
)

additional_data = spark.createDataFrame(
    [(3, "Linus", "platform")],
    ["id", "name", "team"],
)
additional_data.write.format("delta").mode("append").saveAsTable(source_table)

print("New source rows:")
spark.table(source_table).orderBy("id").show()

run_streaming_step("Incremental batch")

New source rows:
+---+------------+-----------+
| id|        name|       team|
+---+------------+-----------+
|  1|Ada Lovelace|engineering|
|  2|       Grace|       data|
|  3|       Linus|   platform|
+---+------------+-----------+

Incremental batch - CDF offsets:
  startOffset: {"sourceVersion":1,"reservoirId":"6b81c71f-bd90-4a0c-8bc0-07cc0678aea1","reservoirVersion":1,"index":-1,"isStartingVersion":false}
  endOffset: {"sourceVersion":1,"reservoirId":"6b81c71f-bd90-4a0c-8bc0-07cc0678aea1","reservoirVersion":3,"index":-1,"isStartingVersion":false}
  checkpoint resume watermark: {"sourceVersion":1,"reservoirId":"6b81c71f-bd90-4a0c-8bc0-07cc0678aea1","reservoirVersion":3,"index":-1,"isStartingVersion":false}
Incremental batch - target table:
+---+------------+-----------+
| id|        name|       team|
+---+------------+-----------+
|  1|Ada Lovelace|engineering|
|  2|       Grace|       data|
|  3|       Linus|   platform|
+---+------------+-----------+



In [4]:
spark.stop()